# CARS NRB Suppression: Transformer Benchmark Evaluation

Compares five deep learning models for extracting the pure Raman signal from CARS spectra
by suppressing the Non-Resonant Background (NRB):

| # | Model | Reference |
|---|-------|-----------|
| 1 | **SpecNet** (CNN baseline) | Paper 1/4 |
| 2 | **Bi-LSTM** (current best) | Paper 2 |
| 3 | **Transformer-Keras** (multi-head attn, 6 blocks) | Thesis Ch.6 |
| 4 | **Transformer-PosEnc** (+ sinusoidal pos.enc + CNN embed) | Thesis Ch.6 |
| 5 | **Transformer-Fourier** (+ Fourier output head) | Thesis Ch.6 |

Evaluation: 300 synthetic test spectra (seed=42) + 4 experimental CARS datasets.

**Before running:** set paths in the Configuration cell below.

In [ ]:
import os

# ------------------------------------------------------------------ #
# PATHS -- edit before running (local Windows or Colab /content/...)  #
# ------------------------------------------------------------------ #

# Directory containing the Transformer weight files from thesis/06_Transformers/Finals/
WEIGHTS_DIR = r"C:\Users\alisa\DS-ML-DL\_other stuff\PhD\thesis_DeepLearning _for_[Raman-CARS]_Spectroscopy\06_Transformers\Finals"

# Directory with 4 experimental CARS/Raman CSV pairs
EXP_DIR = os.path.join(WEIGHTS_DIR, "CARS_experimentalSpectra")

# Bi-LSTM weights from Paper 2 (Vibrational-Spectroscopy-Research repo)
BILSTM_WEIGHTS = r"C:\Users\alisa\DS-ML-DL\Vibrational-Spectroscopy-Research\Four_DL_models_comparison\Bi_LSTM_model_weights.h5"

# Weight filenames inside WEIGHTS_DIR
W_SPECNET          = "weights_SpecNet.h5"
W_TRANSFORMER_K    = "weights_Transformer_timeseriesKeras_2_2.h5"
W_TRANSFORMER_PE   = "weights_Transformer_timeseriesPosEncEmb.h5"
W_TRANSFORMER_FH   = "weights_Transformer_timeseriesFourierHead.h5"  # or _2.h5 if present

print("Paths configured.")
for label, p in [("WEIGHTS_DIR", WEIGHTS_DIR), ("EXP_DIR", EXP_DIR), ("BILSTM_WEIGHTS", BILSTM_WEIGHTS)]:
    exists = os.path.exists(p)
    print(f"  {'OK' if exists else 'MISSING'} {label}: {p}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.metrics as metrics
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Conv1D, Flatten, BatchNormalization, Activation, Dropout,
    LSTM, Bidirectional, TimeDistributed
)

print(f"TensorFlow {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {gpus if gpus else 'None (CPU mode)'}")

## Data Generator

Synthetic CARS spectra: Lorentzian resonances + polynomial NRB + small noise.  
Identical to the generator used for training all models.

In [ ]:
max_features = 15
N_POINTS = 640
nu = np.linspace(0, 1, N_POINTS)


def random_chi3():
    n_lor = np.random.randint(1, max_features)
    a = np.random.uniform(0, 1, n_lor)
    w = np.random.uniform(0, 1, n_lor)
    g = np.random.uniform(0.001, 0.008, n_lor)
    return np.c_[a, w, g]


def build_chi3(params):
    chi3 = np.sum(
        params[:, 0] / (-nu[:, np.newaxis] + params[:, 1] - 1j * params[:, 2]),
        axis=1
    )
    return chi3 / np.max(np.abs(chi3))


def generate_nrb():
    r2, r4, r5 = np.random.randint(-10, 10, size=3)
    r1, r3 = np.random.uniform(-1, 1, size=2)
    nrb = np.polyval([r1, r2, r3, r4, r5], nu)
    nrb -= nrb.min()
    return nrb / nrb.max()


def get_spectrum():
    chi3 = build_chi3(random_chi3()) * np.random.uniform(0.3, 1)
    nrb = generate_nrb()
    noise = np.random.randn(N_POINTS) * np.random.uniform(0.0005, 0.003)
    cars = (np.abs(chi3 + nrb) ** 2) / 2 + noise
    return cars, chi3.imag


def generate_batch(size=1, seed=None):
    if seed is not None:
        np.random.seed(seed)
    X = np.empty((size, N_POINTS, 1))
    y = np.empty((size, N_POINTS))
    for i in range(size):
        X[i, :, 0], y[i, :] = get_spectrum()
    return X, y


# Quick visual check
np.random.seed(0)
x_demo, y_demo = generate_batch(1)
plt.figure(figsize=(12, 4))
plt.plot(nu, x_demo[0, :, 0], label='CARS (input)')
plt.plot(nu, y_demo[0], label='Raman (target)')
plt.legend()
plt.title('Sample synthetic spectrum')
plt.tight_layout()
plt.show()

## Model Architectures

In [ ]:
# SpecNet (CNN baseline, Paper 1)
def build_specnet():
    tf.keras.backend.clear_session()
    m = Sequential([
        BatchNormalization(input_shape=(N_POINTS, 1)),
        Activation('relu'),
        Conv1D(128, kernel_size=32, activation='relu'),
        Conv1D(64,  kernel_size=16, activation='relu'),
        Conv1D(16,  kernel_size=8,  activation='relu'),
        Conv1D(16,  kernel_size=8,  activation='relu'),
        Conv1D(16,  kernel_size=8,  activation='relu'),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=0, l2=0.1)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=0, l2=0.1)),
        Flatten(),
        Dropout(0.25),
        Dense(N_POINTS, activation='relu'),
    ])
    m.compile(loss='mse', optimizer='Adam', metrics=['mae', 'mse'])
    return m


print('SpecNet defined.')

In [ ]:
# Bi-LSTM (best model, Paper 2)
def build_bilstm():
    opt = keras.optimizers.Adam(learning_rate=0.005)
    n_units = 30
    rdrop = 0.16
    drop  = 0.14
    m = Sequential([
        Bidirectional(LSTM(n_units, activation='tanh', dropout=0.0,
                           recurrent_dropout=rdrop, recurrent_activation='sigmoid',
                           return_sequences=True, implementation=1),
                      input_shape=(N_POINTS, 1)),
        Bidirectional(LSTM(n_units, activation='tanh', dropout=drop,
                           recurrent_dropout=rdrop, recurrent_activation='sigmoid',
                           return_sequences=True, implementation=1)),
        Bidirectional(LSTM(n_units, activation='tanh', dropout=drop,
                           recurrent_dropout=rdrop, recurrent_activation='sigmoid',
                           return_sequences=True, implementation=1)),
        TimeDistributed(Dense(1, activation='tanh')),
    ])
    m.compile(loss='mse', optimizer=opt, metrics=['mae', 'mse'])
    return m


print('Bi-LSTM defined.')

In [ ]:
# Transformer-Keras (multi-head attention, 6 blocks, two-layer FFN with BN)
def _encoder_keras(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs

    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim[0], kernel_size=5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters=ff_dim[1], kernel_size=3, padding='same', activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res


def build_transformer_keras():
    inputs = keras.Input(shape=(N_POINTS, 1))
    x = inputs
    for _ in range(6):
        x = _encoder_keras(x, head_size=64, num_heads=16, ff_dim=[128, 64], dropout=0.25)
    x = layers.GlobalAveragePooling1D(data_format='channels_first')(x)
    for dim in [256, 128]:
        x = layers.Dense(dim, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(N_POINTS, activation='relu')(x)
    m = keras.Model(inputs, outputs)
    m.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-4),
              metrics=['mean_absolute_error', 'mse', 'cosine_similarity'])
    return m


print('Transformer-Keras defined.')

In [ ]:
# Transformer-PosEnc (sinusoidal positional encoding + CNN embedding, 6 blocks)

class PositionalEncoding(layers.Layer):
    def __init__(self, embed_dim, max_len=N_POINTS, **kwargs):
        super().__init__(**kwargs)
        positions = np.arange(max_len)[:, np.newaxis]
        dimensions = np.arange(embed_dim)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / embed_dim)
        angle_rads = positions * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        self._pe = tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        return x + self._pe[:, :tf.shape(x)[1], :]

    def get_config(self):
        return super().get_config()


class CNNEmbedding(layers.Layer):
    def __init__(self, embed_dim=64, **kwargs):
        super().__init__(**kwargs)
        self.embed = models.Sequential([
            layers.Conv1D(8,  kernel_size=3, padding='same', activation='relu'),
            layers.Conv1D(32, kernel_size=3, padding='same', activation='relu'),
        ])

    def call(self, x):
        x = tf.transpose(x, perm=[0, 2, 1])
        x = self.embed(x)
        return tf.transpose(x, perm=[0, 2, 1])

    def get_config(self):
        return super().get_config()


def _encoder_pe(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs

    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res


def build_transformer_posenc():
    inputs = keras.Input(shape=(N_POINTS, 1))
    x = CNNEmbedding(embed_dim=64)(inputs)
    x = PositionalEncoding(embed_dim=x.shape[-1])(x)
    for _ in range(6):
        x = _encoder_pe(x, head_size=256, num_heads=8, ff_dim=64, dropout=0.1)
    x = layers.GlobalAveragePooling1D(data_format='channels_first')(x)
    for dim in [256, 128, 64]:
        x = layers.Dense(dim, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(N_POINTS, activation='linear')(x)
    m = keras.Model(inputs, outputs)
    m.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-4),
              metrics=['mean_absolute_error', 'mse', 'cosine_similarity'])
    return m


print('Transformer-PosEnc defined.')

In [ ]:
# Transformer-Fourier (learnable Fourier output head)

class FourierHead(layers.Layer):
    def __init__(self, input_dim, output_dim, num_frequencies=42, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_frequencies = num_frequencies

    def build(self, input_shape):
        self.frequencies = self.add_weight(
            shape=(self.input_dim, self.num_frequencies),
            initializer='random_normal', trainable=True, name='frequencies')
        self.projection = self.add_weight(
            shape=(self.num_frequencies * 2, self.output_dim),
            initializer='random_normal', trainable=True, name='projection')
        super().build(input_shape)

    def call(self, inputs):
        x = tf.matmul(inputs, self.frequencies)
        fourier_features = tf.concat([tf.sin(x), tf.cos(x)], axis=-1)
        return tf.matmul(fourier_features, self.projection)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'input_dim': self.input_dim, 'output_dim': self.output_dim,
                    'num_frequencies': self.num_frequencies})
        return cfg


def _encoder_fh(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs

    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res


def build_transformer_fourier():
    inputs = keras.Input(shape=(N_POINTS, 1))
    x = inputs
    for _ in range(4):
        x = _encoder_fh(x, head_size=256, num_heads=4, ff_dim=4, dropout=0.25)
    x = layers.GlobalAveragePooling1D(data_format='channels_first')(x)
    for dim in [128]:
        x = layers.Dense(dim, activation='relu')(x)
        x = layers.Dropout(0.4)(x)
    outputs = FourierHead(input_dim=128, output_dim=N_POINTS, num_frequencies=42)(x)
    m = keras.Model(inputs, outputs)
    m.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-4),
              metrics=['mean_absolute_error', 'mse', 'cosine_similarity'])
    return m


print('Transformer-Fourier defined.')

## Build Models and Load Weights

In [ ]:
# Build all models
specnet          = build_specnet()
bilstm           = build_bilstm()
transformer_k    = build_transformer_keras()
transformer_pe   = build_transformer_posenc()
transformer_fh   = build_transformer_fourier()

# Load weights (run a forward pass first to initialise lazy layers)
_dummy_X = np.zeros((1, N_POINTS, 1))
for _m in [specnet, bilstm, transformer_k, transformer_pe, transformer_fh]:
    _m.predict(_dummy_X, verbose=0)

def _w(fname):
    return os.path.join(WEIGHTS_DIR, fname)

specnet.load_weights(_w(W_SPECNET))
bilstm.load_weights(BILSTM_WEIGHTS)
transformer_k.load_weights(_w(W_TRANSFORMER_K))
transformer_pe.load_weights(_w(W_TRANSFORMER_PE))
transformer_fh.load_weights(_w(W_TRANSFORMER_FH))

print("All weights loaded.")

MODELS = {
    'SpecNet':          specnet,
    'Bi-LSTM':          bilstm,
    'Transformer-K':    transformer_k,
    'Transformer-PE':   transformer_pe,
    'Transformer-FH':   transformer_fh,
}

## Evaluation on 300 Synthetic Test Spectra

In [ ]:
N_TEST = 300
np.random.seed(42)
X_test, y_test = generate_batch(N_TEST)

results = {}

for name, model in MODELS.items():
    print(f"Evaluating {name}...", end=' ', flush=True)
    y_pred = model.predict(X_test, batch_size=32, verbose=0)
    # Bi-LSTM outputs (batch, 640, 1) -> (batch, 640)
    if y_pred.ndim == 3:
        y_pred = y_pred.squeeze(-1)
    mse_vals = [metrics.mean_squared_error(y_test[i], y_pred[i]) for i in range(N_TEST)]
    mae_vals = [metrics.mean_absolute_error(y_test[i], y_pred[i]) for i in range(N_TEST)]
    results[name] = {
        'mse_mean': np.mean(mse_vals),
        'mse_std':  np.std(mse_vals),
        'mae_mean': np.mean(mae_vals),
        'mae_std':  np.std(mae_vals),
        'mse_vals': mse_vals,
    }
    print(f"MSE={np.mean(mse_vals):.4e} +/- {np.std(mse_vals):.4e}")

print("\nDone.")

In [ ]:
# Benchmark table
rows = []
for name, r in results.items():
    rows.append({
        'Model': name,
        'MSE mean': f"{r['mse_mean']:.4e}",
        'MSE std':  f"{r['mse_std']:.4e}",
        'MAE mean': f"{r['mae_mean']:.4e}",
        'MAE std':  f"{r['mae_std']:.4e}",
    })

df = pd.DataFrame(rows).set_index('Model')
print("=" * 70)
print(f"{'Benchmark: 300 synthetic test spectra (seed=42)':^70}")
print("=" * 70)
print(df.to_string())
print("=" * 70)

# Best model
best = min(results, key=lambda k: results[k]['mse_mean'])
baseline_mse = results['SpecNet']['mse_mean']
bilstm_mse   = results['Bi-LSTM']['mse_mean']
print(f"\nBest model: {best}")
print(f"Improvement over SpecNet: {baseline_mse / results[best]['mse_mean']:.1f}x lower MSE")
print(f"Improvement over Bi-LSTM: {bilstm_mse / results[best]['mse_mean']:.2f}x lower MSE")

In [ ]:
# Per-sample MSE across 300 test spectra
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

colors = ['green', 'orange', 'steelblue', 'purple', 'crimson']
x_axis = np.arange(N_TEST)

ax = axes[0]
for (name, r), c in zip(results.items(), colors):
    ax.plot(x_axis, r['mse_vals'], color=c, alpha=0.7, linewidth=0.9, label=name)
ax.set_xlabel('Test sample index', fontweight='bold')
ax.set_ylabel('MSE', fontweight='bold')
ax.set_title('Per-sample MSE: all models', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

ax = axes[1]
bar_x = np.arange(len(results))
means = [r['mse_mean'] for r in results.values()]
stds  = [r['mse_std']  for r in results.values()]
bars = ax.bar(bar_x, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
ax.set_xticks(bar_x)
ax.set_xticklabels(list(results.keys()), fontweight='bold')
ax.set_ylabel('Mean MSE (+/- std)', fontweight='bold')
ax.set_title('Mean MSE comparison (300 synthetic spectra)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_mse_comparison_synthetic.pdf', bbox_inches='tight')
plt.show()
print("Saved fig_mse_comparison_synthetic.pdf")

In [ ]:
# Visual prediction comparison on 3 random test spectra
np.random.seed(42)
sample_idxs = np.random.choice(N_TEST, 3, replace=False)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
colors = ['green', 'orange', 'steelblue', 'purple', 'crimson']

for row, idx in enumerate(sample_idxs):
    ax = axes[row]
    ax.plot(nu, y_test[idx], 'k-', linewidth=2, label='Ground truth (Raman)', zorder=5)
    for (name, model), c in zip(MODELS.items(), colors):
        x_in = X_test[idx:idx+1]
        y_pred = model.predict(x_in, verbose=0)
        if y_pred.ndim == 3:
            y_pred = y_pred.squeeze(-1)
        ax.plot(nu, y_pred[0], color=c, linewidth=0.9, alpha=0.8, label=name)
    ax.set_title(f'Test sample {idx}', fontweight='bold')
    ax.set_xlabel('Wavenumber (norm.)')
    ax.set_ylabel('Intensity')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_predictions_synthetic.pdf', bbox_inches='tight')
plt.show()
print("Saved fig_predictions_synthetic.pdf")

## Evaluation on Experimental CARS Spectra

Four real CARS datasets from the LUT University spectroscopy lab.

In [ ]:
EXPERIMENTS = [
    ('exp_1_DMPC_lipid',      'DMPC lipid membrane'),
    ('exp_2_MEC_Yeast',       'MEC Yeast cells'),
    ('exp_3_AMP-ADP-ATP',     'AMP-ADP-ATP nucleotides'),
    ('exp_4_ProteinDroplet',  'Protein droplet'),
]

exp_results = {}

fig, axes = plt.subplots(len(EXPERIMENTS), 1, figsize=(14, 5 * len(EXPERIMENTS)))

for row, (key, label) in enumerate(EXPERIMENTS):
    cars_df  = pd.read_csv(os.path.join(EXP_DIR, f"{key}_CARS.csv"),  header=None)
    raman_df = pd.read_csv(os.path.join(EXP_DIR, f"{key}_Raman.csv"), header=None)

    x_exp = np.empty((1, N_POINTS, 1))
    y_exp = np.empty((1, N_POINTS))
    x_exp[0, :, 0] = cars_df[1]
    y_exp[0]        = raman_df[1]
    wn = cars_df[0].values

    ax = axes[row]
    ax.plot(wn, y_exp[0], 'k-', linewidth=2.5, label='Measured Raman', zorder=6)
    ax.plot(wn, x_exp[0, :, 0], 'k--', linewidth=1, alpha=0.4, label='CARS input')

    sample_mse = {}
    colors = ['green', 'orange', 'steelblue', 'purple', 'crimson']
    for (name, model), c in zip(MODELS.items(), colors):
        y_pred = model.predict(x_exp, verbose=0)
        if y_pred.ndim == 3:
            y_pred = y_pred.squeeze(-1)
        mse_v = metrics.mean_squared_error(y_exp[0], y_pred[0])
        sample_mse[name] = mse_v
        ax.plot(wn, y_pred[0], color=c, linewidth=1, alpha=0.85,
                label=f"{name} (MSE={mse_v:.3e})")

    exp_results[label] = sample_mse
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Wavenumber (cm$^{-1}$)')
    ax.set_ylabel('Intensity')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_predictions_experimental.pdf', bbox_inches='tight')
plt.show()
print("Saved fig_predictions_experimental.pdf")

In [ ]:
# Experimental results table
df_exp = pd.DataFrame(exp_results).T
df_exp.index.name = 'Experiment'

print("=" * 90)
print(f"{'Benchmark: Experimental CARS spectra (MSE)':^90}")
print("=" * 90)
print(df_exp.applymap(lambda v: f"{v:.4e}").to_string())
print("=" * 90)

# Which model wins on most experiments?
winners = df_exp.idxmin(axis=1)
print("\nBest model per experiment:")
for exp, winner in winners.items():
    print(f"  {exp}: {winner}")

## Results Summary

Fill in after running:

| Metric | SpecNet | Bi-LSTM | Transformer-K | Transformer-PE | Transformer-FH |
|--------|---------|---------|---------------|----------------|----------------|
| MSE (synthetic, mean) | — | — | — | — | — |
| MAE (synthetic, mean) | — | — | — | — | — |
| Wins on exp. spectra  | — | — | — | — | — |

**Next steps:**
- If a Transformer variant beats Bi-LSTM: this becomes the basis for Paper 5.
- Key narrative: *First application of a Transformer to CARS NRB suppression.*
- Figures to include: `fig_mse_comparison_synthetic.pdf`, `fig_predictions_synthetic.pdf`, `fig_predictions_experimental.pdf`